In [69]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, AdaBoostClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CLUSTER_DIR = os.path.join(DATA_DIR, 'clusters')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
SUBGROUP_MODEL_DIR = os.path.join(MODEL_DIR, 'subgroup_models')
os.makedirs(SUBGROUP_MODEL_DIR, exist_ok=True)

In [70]:
def custom_accuracy(y_true, y_pred):
    """Project metric: acc = TT / (TT + TF) = recall of bankrupt class.
    TT = correctly predicted bankrupt, TF = bankrupt predicted as non-bankrupt."""
    tt = ((y_true == 1) & (y_pred == 1)).sum()
    tf = ((y_true == 1) & (y_pred == 0)).sum()
    if tt + tf == 0:
        return 0.0
    return tt / (tt + tf)

def sparsity_check(y_pred, threshold=0.20):
    """Check if < 20% of predictions are bankrupt."""
    rate = y_pred.mean()
    return rate < threshold, rate

def evaluate_model(y_true, y_pred, label=''):
    """Print evaluation metrics."""
    acc = custom_accuracy(y_true, y_pred)
    ok, rate = sparsity_check(y_pred)
    cm = confusion_matrix(y_true, y_pred)
    print(f'--- {label} ---')
    print(f'Custom Accuracy (TT/(TT+TF)): {acc:.4f}')
    print(f'Sparsity: {rate:.4f} ({"PASS" if ok else "FAIL"})')
    print(f'Confusion Matrix:\n{cm}')
    print(f'Classification Report:\n{classification_report(y_true, y_pred, zero_division=0)}')
    return acc, rate

In [71]:
# Load cluster 3 data
df3 = pd.read_csv(os.path.join(CLUSTER_DIR, 'cluster_3.csv'))
y3 = df3['Bankrupt?'].values
X3 = df3.drop(columns=['Bankrupt?'])
feature_names_3 = df3.drop(columns=['Bankrupt?']).columns.tolist()

print(f'Cluster 3: {X3.shape[0]} samples, {X3.shape[1]} features')
print(f'Bankrupt: {y3.sum()} ({y3.mean():.4f})')

Cluster 3: 1707 samples, 95 features
Bankrupt: 7 (0.0041)


In [132]:
mi_scores_3 = mutual_info_classif(X3, y3, random_state=RANDOM_STATE)
mi_series_3 = pd.Series(mi_scores_3, index=feature_names_3).sort_values(ascending=False)
print('Top 15 features by Mutual Information:')
for i, (feat, score) in enumerate(mi_series_3.head(15).items()):
    print(f'  {i+1:2d}. {feat[:55]:55s}  MI = {score:.4f}')
corr_matrix = X3.corr().abs()
CORR_THRESH = 0.85
selected = []
for feat in mi_series_3.index:
    if not selected or not any(corr_matrix.loc[feat, s] > CORR_THRESH for s in selected):
        selected.append(feat)
    if len(selected) >= 10:
        break

X3_sel = df3[selected].values

Top 15 features by Mutual Information:
   1. Net Income Flag                                          MI = 0.0306
   2. Contingent liabilities/Net worth                         MI = 0.0193
   3. Net Value Per Share (A)                                  MI = 0.0036
   4. Total Asset Turnover                                     MI = 0.0034
   5. Net Value Per Share (B)                                  MI = 0.0034
   6. Equity to Long-term Liability                            MI = 0.0031
   7. Net Worth Turnover Rate (times)                          MI = 0.0030
   8. Total expense/Assets                                     MI = 0.0030
   9. Fixed Assets to Assets                                   MI = 0.0026
  10. Revenue Per Share (Yuan ¥)                               MI = 0.0022
  11. Long-term fund suitability ratio (A)                     MI = 0.0021
  12. Current Liabilities/Liability                            MI = 0.0021
  13. Current Liability to Liability                         

In [123]:
base_ms = [
    ('rf', RandomForestClassifier(
        n_estimators=100,
        max_depth=4,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )),
    ('gb', GradientBoostingClassifier(
        n_estimators=80,
        max_depth=3,
        learning_rate=0.05,
        min_samples_leaf=10,
        random_state=RANDOM_STATE
    )),
    ('et', ExtraTreesClassifier(
        n_estimators=100,
        max_depth=4,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )),
]



In [130]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
stack = StackingClassifier(
    estimators=base_ms,
    final_estimator=LogisticRegression(max_iter=1000, C=0.5, class_weight="balanced", random_state=RANDOM_STATE),
    cv=cv,
    stack_method="predict_proba",
    n_jobs=-1
    )


selected = []


for feat in mi_series_3.index:
    if not selected or not any(corr_matrix.loc[feat, s] > CORR_THRESH for s in selected):
        selected.append(feat)
    if len(selected) >= 10:
        break

X_sel = df3[selected].values

print(f"\n=== Testing top {10} features ===")
print("Features:", selected)

probs = cross_val_predict(
    stack,
    X_sel,
    y3,
    cv=cv,
    method="predict_proba"
)[:, 1]


thresholds = np.arange(0.05, 0.70, 0.01)

best = None

for t in thresholds:
    preds = (probs >= t).astype(int)

    sparsity = preds.mean()
    tt = ((y3 == 1) & (preds == 1)).sum()
    tf = ((y3 == 1) & (preds == 0)).sum()

    if sparsity <= 0.20:
        score = tt / (tt + tf + 1e-9)

        if best is None or score > best["score"]:
            best = {"t": t, "score": score, "sparsity": sparsity}

best_t = best["t"]

final_preds = (probs >= best_t).astype(int)

evaluate_model(y3, final_preds, label=f"Final model (t={best_t:.3f})")


=== Testing top 10 features ===
Features: ['Net Income Flag', 'Contingent liabilities/Net worth', 'Net Value Per Share (A)', 'Total Asset Turnover', 'Equity to Long-term Liability', 'Total expense/Assets', 'Fixed Assets to Assets', 'Revenue Per Share (Yuan ¥)', 'Long-term fund suitability ratio (A)', 'Current Liabilities/Liability']
--- Final model (t=0.550) ---
Custom Accuracy (TT/(TT+TF)): 0.5714
Sparsity: 0.1851 (PASS)
Confusion Matrix:
[[1388  312]
 [   3    4]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.82      0.90      1700
           1       0.01      0.57      0.02         7

    accuracy                           0.82      1707
   macro avg       0.51      0.69      0.46      1707
weighted avg       0.99      0.82      0.89      1707



(np.float64(0.5714285714285714), np.float64(0.18512009373169303))

In [133]:
final_model = StackingClassifier(
    estimators=base_ms,
    final_estimator=LogisticRegression(
        class_weight='balanced',
        C=0.1,
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    cv=3,
    stack_method='predict_proba',
    n_jobs=-1
)

final_model.fit(X_sel, y3)
train_proba = final_model.predict_proba(X_sel)[:, 1]
train_preds = (train_proba >= best_t).astype(int)

train_results = evaluate_model(
    y3,
    train_preds,
    label=f'Cluster 3 — In-sample diagnostic only (thresh={best_t:.2f})'
)

--- Cluster 3 — In-sample diagnostic only (thresh=0.55) ---
Custom Accuracy (TT/(TT+TF)): 1.0000
Sparsity: 0.1740 (PASS)
Confusion Matrix:
[[1410  290]
 [   0    7]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.83      0.91      1700
           1       0.02      1.00      0.05         7

    accuracy                           0.83      1707
   macro avg       0.51      0.91      0.48      1707
weighted avg       1.00      0.83      0.90      1707

